# Вариант 1: классификация стиля речи

Классы: **0 — разговорный, 1 — официальный**. Основная модель — TF-IDF по словам и символам + LogisticRegression. Подлинность голоса не классифицируется.

Код обучения общий с приложением: `DZ_03.train_classifier`. CNN и ансамбль не входят в этот вариант. Исторические метрики не считаются результатами нового разбиения. Не сохраняйте в Git выводы с коммерческими текстами.


## 1. Подготовка

Сначала выполните команды скачивания и подготовки из `DZ_03/RUNBOOK.md`. Для ноутбука: `uv sync --locked --extra notebook --extra training`; выберите Python kernel из `.venv`.


In [ ]:
from pathlib import Path
import sys
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

ROOT = Path.cwd().resolve()
if not (ROOT / "pyproject.toml").is_file():
    ROOT = ROOT.parent
if not (ROOT / "pyproject.toml").is_file():
    raise RuntimeError("Откройте ноутбук из корня проекта или DZ_03")
sys.path.insert(0, str(ROOT))
from DZ_03.train_classifier import load_splits, train_and_export
from DZ_03.compare_experiments import compare
DATA = ROOT / "data/processed"


## 2. Проверка выборок

Проверяем отсутствие пересечений текстов и доступных идентификаторов записей/дикторов. Метки по доменам являются слабыми и не заменяют ручную оценку стиля.


In [ ]:
splits = load_splits(DATA)
pd.DataFrame({name: {"rows": len(frame), "class_0": int((frame.label == 0).sum()),
                    "class_1": int((frame.label == 1).sum()),
                    "groups": frame.group_id.nunique() if "group_id" in frame else None}
              for name, frame in splits.items()}).T


## 3. Основной классификатор

Слова 1–3-граммы и символы 3–5-граммы; `class_weight="balanced"`, random_state=42. Векторизаторы обучаются только на train. Постоянный классификатор служит общей точкой сравнения.


In [ ]:
MODEL = ROOT / "models/style_classifier.joblib"
metrics = train_and_export(DATA, MODEL)
pd.DataFrame([{ "split": split, "method": method,
               **{key: scores[key] for key in ("accuracy", "balanced_accuracy", "macro_f1")}}
              for split, methods in metrics.items() for method, scores in methods.items()])


## 4. Ошибки по классам

Confusion matrix: строки — истинный класс, столбцы — предсказанный. Одна accuracy при дисбалансе недостаточна.


In [ ]:
sns.heatmap(metrics["test"]["model"]["confusion_matrix"], annot=True, fmt="d", cmap="Blues",
            xticklabels=["0: conversational", "1: official"],
            yticklabels=["0: conversational", "1: official"])
plt.xlabel("Predicted")
plt.ylabel("True")
plt.show()
pd.DataFrame(metrics["test"]["model"]["classification_report"]).T


## 5. Усложнение с TTS

Создайте `manifest.csv` по командам RUNBOOK. Для синтеза используются только train-фразы обоих классов; затем аудио распознаётся Whisper. Метка означает стиль, `is_synthetic` — происхождение. Validation/test не меняются.

Ниже сравниваются одинаковые классификаторы, различается только train. Модель выбирается по validation macro-F1; test не участвует в выборе. Эта ячейка не заменяет модель приложения. Для замены используйте CLI с `--deploy`, описанный в RUNBOOK.


In [ ]:
manifest = ROOT / "data/tts/run1/manifest.csv"
if manifest.is_file():
    comparison = compare(DATA, manifest, ROOT / "models/experiments")
    print("Кандидат по validation:", comparison["selected"])
    display(pd.DataFrame([{ "experiment": experiment, "split": split,
                            **{key: values["model"][key] for key in ("accuracy", "balanced_accuracy", "macro_f1")}}
                         for experiment, experiments in comparison["metrics"].items()
                         for split, values in experiments.items()]))
else:
    print("TTS ещё не выполнен. Следуйте DZ_03/RUNBOOK.md; новые метрики пока не заявляются.")


## 6. Выводы для отчёта

Перенесите фактические размеры выборок и результаты из `models/style_classifier.metrics.json` и `models/experiments/comparison.md` в `Отчёт.md`. Отдельно опишите изменения метрик класса 0, влияние TTS и ограничения доменной разметки. Улучшение после TTS не гарантируется. Для независимой проверки нужны вручную размеченные реальные голосовые.
